# 04 — Window Functions

Purpose:
Practice SQL window functions using PostgreSQL telemetry data.

This notebook covers:
- PostgreSQL connection
- smoke test
- helper function to run SQL
- table inspection helper
- ROW_NUMBER
- RANK
- DENSE_RANK
- LAG
- LEAD
- moving averages
- PARTITION BY
- ORDER BY inside OVER()
- capacity and telemetry interview explanations


## Cell 2 — Install/import dependencies


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")


## Cell 3 — Connection settings


In [ ]:
DB_HOST = "host.docker.internal"
DB_PORT = 5432
DB_NAME = "studybook"
DB_USER = "sb_user"
DB_PASSWORD = "sb_pass_123"

password_encoded = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{password_encoded}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database URL created.")

# If this notebook runs directly on Windows instead of inside a container,
# change DB_HOST to "localhost".


## Cell 4 — Smoke test connection


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, now();"))
    row = result.fetchone()

row


## Cell 5 — Helper function to run SQL


In [ ]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local PostgreSQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)


## Cell 6 — Helper function to inspect one table safely


In [ ]:
def inspect_table_safe(table_name: str) -> None:
    """
    Safely inspect a table without changing data.
    Shows column metadata, row count, and a small preview.
    """
    metadata_sql = f"""
    SELECT
        table_schema,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = '{table_name}'
    ORDER BY ordinal_position;
    """

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM {table_name};
    """

    preview_sql = f"""
    SELECT *
    FROM {table_name}
    LIMIT 10;
    """

    print(f"Column metadata for public.{table_name}")
    display(run_sql(metadata_sql))

    print(f"Row count for public.{table_name}")
    display(run_sql(count_sql))

    print(f"Preview rows from public.{table_name}")
    display(run_sql(preview_sql))


# 04 — Window Function Practice


## 04.1 Verify tables used in this notebook

This notebook uses:
- `telemetry_samples` as the metric fact table
- `services` as the service lookup table
- `hosts` as the host lookup table when needed


In [ ]:
sql = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_name IN ('telemetry_samples', 'services', 'hosts')
ORDER BY table_name;
"""

run_sql(sql)


## 04.2 Inspect telemetry_samples

Window functions work well on timestamped telemetry because we often want
row-level detail plus comparisons, ranks, or moving averages.


In [ ]:
inspect_table_safe("telemetry_samples")


## 04.3 Preview telemetry with service names

Before using window functions, preview the base joined dataset.
This keeps raw rows visible instead of collapsing them like GROUP BY.


In [ ]:
sql = """
SELECT
    t.sampled_at,
    s.service_name,
    t.host_id,
    t.cpu_utilization_pct,
    t.memory_utilization_pct,
    t.p95_latency_ms,
    t.requests_per_min,
    t.error_rate_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.host_id,
    t.sampled_at
LIMIT 30;
"""

run_sql(sql)


## 04.4 ROW_NUMBER by service

ROW_NUMBER gives each row a sequence number inside a partition.
Here, each service gets its own timeline numbering.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    ROW_NUMBER() OVER (
        PARTITION BY s.service_name
        ORDER BY t.sampled_at
    ) AS row_number_within_service
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    row_number_within_service
LIMIT 50;
"""

run_sql(sql)


## 04.5 RANK services by CPU sample

RANK orders rows by a metric. Here we rank individual telemetry samples by CPU utilization.
RANK may skip numbers when there are ties.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    RANK() OVER (
        ORDER BY t.cpu_utilization_pct DESC
    ) AS cpu_rank_overall
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY cpu_rank_overall
LIMIT 25;
"""

run_sql(sql)


## 04.6 DENSE_RANK services by CPU sample

DENSE_RANK is similar to RANK, but it does not skip rank numbers after ties.
This is useful for clean leaderboards.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    DENSE_RANK() OVER (
        ORDER BY t.cpu_utilization_pct DESC
    ) AS dense_cpu_rank_overall
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY dense_cpu_rank_overall
LIMIT 25;
"""

run_sql(sql)


## 04.7 Rank hottest sample per service

PARTITION BY changes the ranking scope.
Instead of ranking all rows globally, this ranks rows inside each service.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    RANK() OVER (
        PARTITION BY s.service_name
        ORDER BY t.cpu_utilization_pct DESC
    ) AS cpu_rank_within_service
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    cpu_rank_within_service
LIMIT 50;
"""

run_sql(sql)


## 04.8 Find top CPU sample per service using a CTE

Window functions are often paired with CTEs.
First rank the rows, then filter for rank = 1.


In [ ]:
sql = """
WITH ranked_samples AS (
    SELECT
        s.service_name,
        t.host_id,
        t.sampled_at,
        t.cpu_utilization_pct,
        RANK() OVER (
            PARTITION BY s.service_name
            ORDER BY t.cpu_utilization_pct DESC
        ) AS cpu_rank_within_service
    FROM telemetry_samples t
    JOIN services s
        ON s.service_id = t.service_id
)
SELECT
    service_name,
    host_id,
    sampled_at,
    cpu_utilization_pct,
    cpu_rank_within_service
FROM ranked_samples
WHERE cpu_rank_within_service = 1
ORDER BY service_name;
"""

run_sql(sql)


## 04.9 LAG previous CPU sample

LAG looks backward to a previous row.
This is useful for comparing current telemetry to the prior sample.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    LAG(t.cpu_utilization_pct) OVER (
        PARTITION BY s.service_name, t.host_id
        ORDER BY t.sampled_at
    ) AS previous_cpu_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.host_id,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 04.10 CPU change from previous sample

After using LAG, subtract the previous value from the current value.
This shows whether CPU increased or decreased between samples.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    LAG(t.cpu_utilization_pct) OVER (
        PARTITION BY s.service_name, t.host_id
        ORDER BY t.sampled_at
    ) AS previous_cpu_pct,
    ROUND(
        t.cpu_utilization_pct
        - LAG(t.cpu_utilization_pct) OVER (
            PARTITION BY s.service_name, t.host_id
            ORDER BY t.sampled_at
        ),
        2
    ) AS cpu_change_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.host_id,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 04.11 LEAD next CPU sample

LEAD looks forward to the next row.
This can be useful when comparing current telemetry to the next sample or looking ahead in a timeline.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    LEAD(t.cpu_utilization_pct) OVER (
        PARTITION BY s.service_name, t.host_id
        ORDER BY t.sampled_at
    ) AS next_cpu_pct
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.host_id,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 04.12 Moving average CPU

A moving average smooths noisy telemetry.
This example calculates the average CPU for the current row and two previous rows.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    ROUND(
        AVG(t.cpu_utilization_pct) OVER (
            PARTITION BY s.service_name, t.host_id
            ORDER BY t.sampled_at
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS cpu_moving_avg_3_samples
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.host_id,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 04.13 Moving average latency

The same pattern can smooth latency.
Since p95_latency_ms is already a sampled P95 value, this moving average smooths sampled P95 latency.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.p95_latency_ms,
    ROUND(
        AVG(t.p95_latency_ms) OVER (
            PARTITION BY s.service_name, t.host_id
            ORDER BY t.sampled_at
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ),
        2
    ) AS p95_latency_moving_avg_3_samples
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.host_id,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 04.14 Running total requests by service

A running total accumulates values over time.
This can show request growth through the day for each service.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.sampled_at,
    t.requests_per_min,
    SUM(t.requests_per_min) OVER (
        PARTITION BY s.service_name
        ORDER BY t.sampled_at
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_requests_per_min_total
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 04.15 Window function versus GROUP BY

GROUP BY collapses many rows into fewer summary rows.
Window functions keep the original rows and add calculations beside them.
This is why window functions are useful for telemetry timelines.


In [ ]:
sql = """
SELECT
    s.service_name,
    t.host_id,
    t.sampled_at,
    t.cpu_utilization_pct,
    ROUND(
        AVG(t.cpu_utilization_pct) OVER (
            PARTITION BY s.service_name
        ),
        2
    ) AS service_avg_cpu_visible_on_each_row
FROM telemetry_samples t
JOIN services s
    ON s.service_id = t.service_id
ORDER BY
    s.service_name,
    t.host_id,
    t.sampled_at
LIMIT 50;
"""

run_sql(sql)


## 04.16 Final interview explanation

Window functions are useful when I need row-level telemetry detail plus analytics. GROUP BY collapses rows into summaries, but window functions keep each sample visible while adding calculations like rank, previous value, next value, moving average, or running total. In capacity work, I use PARTITION BY to separate services or hosts, ORDER BY to define the timeline, LAG to compare current and previous samples, RANK to find the hottest services, and moving averages to smooth noisy telemetry.
